# Method: Ridge Regression

#### Algorithm Reference

T. Hastie, R. Tibshirani, and J. Friedman, *The Elements of Statistical Learning: Data Mining, Inference, and Prediction*, 2nd ed. New York, NY, USA: Springer, 2009, sec. 3.4.1.

### I. Imports and Reads

#### Imports

In [30]:
import numpy as np
import csv 
import EvaluationMetrics as em

#### Reads

In [31]:
with open('../dataset/processed_test.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    test_in = np.array(data)
    test_out = np.array(output)

with open('../dataset/processed_tune.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    tune_in = np.array(data)
    tune_out = np.array(output)

with open('../dataset/processed_train.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    train_in = np.array(data)
    train_out = np.array(output)

train_in

array([['5', '0.016057125812515255', '-13.062962962962963',
        '37.596296296296295', '37.00214814814815', '36.408'],
       ['5', '0.016057125812515255', '-12.596296296296295',
        '37.596296296296295', '37.00214814814815', '36.408'],
       ['5', '0.016057125812515255', '-13.596296296296295',
        '37.596296296296295', '37.00214814814815', '36.408'],
       ...,
       ['2', '0.02745569032449328', '0.5259574468084978',
        '37.2540425531915', '36.258539325842705', '36.4595294117647'],
       ['2', '0.02745569032449328', '-4.254042553191503',
        '37.2540425531915', '36.258539325842705', '36.4595294117647'],
       ['2', '0.02745569032449328', '0.07595744680849492',
        '37.2540425531915', '36.258539325842705', '36.4595294117647']],
      shape=(3174, 6), dtype='<U22')

#### Ridge Regression Functions

In [32]:
LAMB = 1  # hyperparameter, the lambda value

def compute_train_stats(X, y):
    x_mean = np.mean(X, axis=0) 			# one mean per column
    x_std = np.std(X, axis=0)				# one std per column
    y_mean = np.mean(y)
    return x_mean, x_std, y_mean

def standardize(X, x_mean, x_std):
    Xs = (X - x_mean) / x_std
    return Xs

def solve_coefficients(lamb, Xs, yc):
    p = Xs.shape[1]
    A = Xs.T @ Xs + lamb * np.eye(p)	  	# XTX + lambI
    b = Xs.T @ yc  							# XTy
    coeff = np.linalg.solve(A, b)
    return coeff

def fit(lamb, X, y):
    # get the training stats
    x_mean, x_std, y_mean = compute_train_stats(X, y)

	# standardize X and center y
    Xs = standardize(X, x_mean, x_std)
    yc = y - y_mean

	# solve for coeffs
    coeff = solve_coefficients(lamb, Xs, yc)
    intercept = y_mean

    return {
        'coeff': coeff,
        'intercept': intercept,
        'x_mean': x_mean,
        'x_std': x_std,
    }

def predict(model, X):
    # standardize X using the training statistics
    Xs = standardize(X, model['x_mean'], model['x_std'])

    # apply the intercept and coefficients
    y_pred = model['intercept'] + Xs @ model['coeff']
    return y_pred

#### Perform Ridge Regression

In [33]:
# convert CSV strings to numbers
X_train = train_in.astype(float)
y_train = train_out.astype(float)
X_tune = tune_in.astype(float)
y_tune = tune_out.astype(float)

# train, the predict
model = fit(LAMB, X_train, y_train)
y_pred = predict(model, X_tune)

#### Display Ridge Regression Predictions

##### May 2020 as starting for test

In [34]:
provinces = ['Metropolitan Manila', 'Nueva Ecija', 'Iloilo', 'South Cotabato', 'Basilan', 'Maguindanao', 'Ifugao', 'Kalinga', 'Ilocos Norte', 'Ilocos Sur', 'Cagayan', 'Isabela',
 'Quirino', 'Bataan', 'Bulacan', 'Batangas', 'Cavite', 'Laguna', 'Quezon', 'Rizal', 'Marinduque', 'Occidental Mindoro', 'Oriental Mindoro', 'Palawan', 'Romblon', 'Zamboanga del Sur',
 'Albay', 'Catanduanes', 'Masbate', 'Sorsogon', 'Aklan', 'Guimaras', 'Negros Occidental', 'Bohol', 'Siquijor', 'Biliran', 'Leyte', 'Northern Samar', 'Samar', 'Southern Leyte', 'Bukidnon',
 'Camiguin', 'Lanao del Norte', 'Misamis Occidental', 'Misamis Oriental', 'Davao Oriental', 'Davao del Norte', 'North Cotabato', 'Sarangani', 'Agusan del Norte', 'Surigao del Norte',
 'Surigao del Sur']

labels = []
with open('../dataset/wfp_food_prices_phl.csv', 'r') as f:
    read = csv.reader(f)
    for x in read:
        if x[0][:4] == '2025' and x[8] == 'Rice (regular, milled)' and x[12] == 'Retail' and x[2] in provinces:
            labels.append({
                'year' : int(x[0][:4]),
                'month' : int(x[0][5:7]),
                'province' : x[2],
                'price' : float(x[14]),
            })

# Stop if the labels do not line up with the 2025 rows
assert len(labels) == len(y_tune)
for i in range(len(labels)):
    assert labels[i]['month'] == X_tune[i, 0]
    assert labels[i]['price'] == y_tune[i]

# Print each prediction next to the correct answer
print(f"{'Month':>5}  {'Province':<22} {'Predicted':>10} {'Actual':>10} {'Error':>8}")
for label, pred, actual in zip(labels, y_pred, y_tune):
    print(f"{label['month']:>5}  {label['province']:<22} {pred:>10.2f} {actual:>10.2f} {pred - actual:>8.2f}")

Month  Province                Predicted     Actual    Error
    1  Basilan                     52.10      50.39     1.71
    1  Maguindanao                 48.24      47.75     0.49
    1  Ifugao                      48.87      48.83     0.04
    1  Kalinga                     50.07      50.88    -0.81
    1  Metropolitan Manila         46.29      44.52     1.77
    1  Ilocos Norte                47.25      43.63     3.62
    1  Ilocos Sur                  47.20      46.23     0.97
    1  Cagayan                     44.99      42.86     2.13
    1  Isabela                     47.07      46.38     0.69
    1  Quirino                     44.57      40.53     4.04
    1  Bataan                      47.10      46.00     1.10
    1  Bulacan                     47.81      44.62     3.19
    1  Bulacan                     47.81      46.73     1.08
    1  Nueva Ecija                 45.84      44.67     1.17
    1  Batangas                    53.50      49.93     3.57
    1  Cavite           

#### Evaluation Metrics on Initial Results

##### May 2020 as starting for test

In [35]:
print(f"MAE:  	₱{em.mae(y_tune, y_pred):.4f}/kg")
print(f"RMSE: 	₱{em.rmse(y_tune, y_pred):.4f}/kg")
print(f"MAPE: 	{em.mape(y_tune, y_pred):.4f}%")
print(f"R2:   	{em.r_squared(y_tune, y_pred):.4f}")

MAE:  	₱1.6587/kg
RMSE: 	₱2.1310/kg
MAPE: 	4.0133%
R2:   	0.7774
